## Modelos

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb
import joblib, os

In [2]:
os.makedirs("../models", exist_ok=True)

train = pd.read_parquet("../data/train_processed.parquet")
test  = pd.read_parquet("../data/test_processed.parquet")

TARGET   = "default"
FEATURES = [c for c in train.columns if c != TARGET]

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"Default rate train : {y_train.mean():.1%}")
print(f"Default rate test  : {y_test.mean():.1%}")

X_train : (451805, 39)
X_test  : (97761, 39)
Default rate train : 20.2%
Default rate test  : 27.2%


In [3]:
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        C=0.1,
        max_iter=1000,
        random_state=42,
        class_weight="balanced"
    ))
])

lr_pipeline.fit(X_train, y_train)
lr_proba = lr_pipeline.predict_proba(X_test)[:, 1]

print("Logística entrenada ✓")
joblib.dump(lr_pipeline, "../models/lr_baseline.pkl")

Logística entrenada ✓


['../models/lr_baseline.pkl']

In [4]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight : {scale_pos_weight:.2f}")

xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="auc",
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
print(f"\nMejor iteración : {xgb_model.best_iteration}")
joblib.dump(xgb_model, "../models/xgb_creditrisk.pkl")

scale_pos_weight : 3.94
[0]	validation_0-auc:0.67718
[50]	validation_0-auc:0.69128
[100]	validation_0-auc:0.69397
[150]	validation_0-auc:0.69490
[200]	validation_0-auc:0.69490
[250]	validation_0-auc:0.69502
[300]	validation_0-auc:0.69511
[350]	validation_0-auc:0.69503
[352]	validation_0-auc:0.69506

Mejor iteración : 302


['../models/xgb_creditrisk.pkl']

In [5]:
np.save("../data/lr_proba.npy",  lr_proba)
np.save("../data/xgb_proba.npy", xgb_proba)
np.save("../data/y_test.npy",    y_test.values)
print("Probabilidades guardadas ✓")

Probabilidades guardadas ✓


In [6]:
print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")

X_train : (451805, 39)
X_test  : (97761, 39)
